# Fine-tune BERTugues (primary) + BERTimbau (baseline) on combined corpora

In [ ]:
!pip -q install 'transformers>=4.44' accelerate datasets scikit-learn pandas pyyaml joblib

In [ ]:
import os, glob, zipfile
with zipfile.ZipFile('repo.zip') as z:
    z.extractall('repo')
os.remove('repo.zip')
hits = glob.glob('repo/**/ml/src/training/train_transformer.py', recursive=True)
assert hits, 'train_transformer.py not found in repo.zip - check the zip contents'
os.chdir(hits[0].split('/ml/')[0])
!rm -rf ml/artifacts .git node_modules frontend/node_modules data/raw
print('cwd:', os.getcwd())

In [ ]:
!python ml/src/data/prepare_fakebr.py
!python ml/src/data/prepare_faketruebr.py --hf-dataset recogna-nlp/fakerecogna2-extrativa --out data/processed/fakerecogna2.csv
!python ml/src/data/prepare_faketruebr.py --hf-dataset recogna-nlp/fakerecogna2-abstrativa --out data/processed/fakerecogna2_abs.csv

In [ ]:
import torch; print('CUDA available:', torch.cuda.is_available())
!python ml/src/training/train_transformer.py
!python ml/src/training/train_transformer.py --model bertimbau


In [ ]:
!python ml/src/data/prepare_faketruebr.py --hf-dataset recogna-nlp/FakeRecogna --invert-label --out data/processed/fakerecogna1.csv
!python ml/src/evaluation/eval_crosscorpus.py --data data/processed/fakerecogna1.csv --sample 5000

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
for name in ['bertugues', 'bertimbau']:
    if os.path.isdir(f'ml/artifacts/{name}'):
        !cd ml/artifacts && zip -r -q /content/drive/MyDrive/{name}.zip {name} -x '{name}/_trainer/*'
        print(f'saved to Drive: {name}.zip')



In [ ]:
from getpass import getpass
import os
os.environ['HUGGINGFACE_API_KEY'] = getpass('HuggingFace write token: ')
!python ml/src/push_to_hub.py
